In [ ]:
%cd drive/MyDrive/captch_training/

/content/drive/MyDrive/captch_training


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CRNN(nn.Module):
    def __init__(self, img_h=50, img_w=200, num_classes=37, hidden_size=256, num_layers=2):
        super(CRNN, self).__init__()

        # CNN Feature Extractor - optimized for 200x50 input
        self.cnn = nn.Sequential(
            # Input: (batch, 1, 50, 200)
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),  # (batch, 64, 25, 100)

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),  # (batch, 128, 12, 50)

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 1), padding=(0, 1)),  # (batch, 256, 6, 51)

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 1), padding=(0, 1)),  # (batch, 512, 3, 52)

            nn.Conv2d(512, 512, kernel_size=(3, 2), padding=(0, 0)),  # (batch, 512, 1, 51)
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )

        # RNN layers
        self.rnn = nn.LSTM(
            input_size=512,  # channels after CNN
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        # Output layer
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        # CNN feature extraction
        x = self.cnn(x)  # Expected output: (batch, 512, 1, 51)

        # Verify dimensions
        batch_size, channels, height, width = x.size()
        assert height == 1, f"Expected height=1 after CNN, got {height}"

        # Reshape for RNN: (batch, channels, 1, width) -> (batch, width, channels)
        x = x.squeeze(2)  # Remove height dimension: (batch, 512, 51)
        x = x.permute(0, 2, 1)  # (batch, 51, 512)

        # RNN sequence modeling
        rnn_out, _ = self.rnn(x)  # (batch, 51, hidden_size*2)

        # Final classification
        output = self.fc(rnn_out)  # (batch, 51, num_classes)

        # Transpose for CTC: (seq_len, batch, num_classes)
        output = output.permute(1, 0, 2)  # (51, batch, num_classes)

        return output

In [ ]:
import cv2
import numpy as np
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class CAPTCHADataset(Dataset):
    def __init__(self, image_paths, labels, char_to_idx, img_height=50, img_width=200):
        self.image_paths = image_paths
        self.labels = labels
        self.char_to_idx = char_to_idx
        self.img_height = img_height
        self.img_width = img_width

        # Transform for 200x50 images
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

    def __len__(self):
        return len(self.image_paths)

    def preprocess_image(self, image_path):
        # Load and convert to grayscale
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Could not load image: {image_path}")

        # Resize to exactly 200x50 (width x height)
        img = cv2.resize(img, (self.img_width, self.img_height), interpolation=cv2.INTER_CUBIC)

        # Optional: Apply contrast enhancement for better features
        # img = cv2.equalizeHist(img)

        return img

    def label_to_tensor(self, label):
        return torch.LongTensor([self.char_to_idx[char] for char in label])

    def __getitem__(self, idx):
        img = self.preprocess_image(self.image_paths[idx])
        label = self.labels[idx]

        img_tensor = self.transform(img)
        label_tensor = self.label_to_tensor(label)

        return img_tensor, label_tensor, len(label)

In [ ]:
# Updated Training Configuration
# Model configuration for 200x50 CAPTCHAs
# IMG_HEIGHT = 50
# IMG_WIDTH = 200

# # Example character set (adjust based on your CAPTCHA)
# # CHARSET = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
# CHARSET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%"
# char_to_idx = {char: idx + 1 for idx, char in enumerate(CHARSET)}  # +1 to reserve 0 for CTC blank
# char_to_idx['<BLANK>'] = 0
# NUM_CLASSES = len(char_to_idx)  # 37 for digits + uppercase letters + blank

# # Initialize model
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = CRNN(
#     img_h=IMG_HEIGHT,
#     img_w=IMG_WIDTH,
#     num_classes=NUM_CLASSES,
#     hidden_size=256,
#     num_layers=2
# ).to(device)

# # CTC Loss
# criterion = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)

# # Optimizer
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

In [ ]:
import torch
import numpy as np

def ctc_greedy_decode(predictions, char_to_idx, blank_index=0):
    """
    predictions: (seq_len, batch, num_classes)
    """
    idx_to_char = {v: k for k, v in char_to_idx.items() if k != '<BLANK>'}

    # Get most probable indices
    _, max_indices = torch.max(predictions, dim=2)  # (seq_len, batch)

    decoded_texts = []
    for b in range(max_indices.size(1)):
        prev_idx = -1
        text = ""
        for t in range(max_indices.size(0)):
            idx = max_indices[t, b].item()
            if idx != blank_index and idx != prev_idx:
                text += idx_to_char.get(idx, '')
            prev_idx = idx
        decoded_texts.append(text)
    return decoded_texts

def ctc_beam_search_decode(predictions, char_to_idx, beam_width=5, blank_index=0):
    """
    Simple beam search for CTC decoding (for production, consider using warp-ctc or flashlight)
    """
    from collections import defaultdict

    idx_to_char = {v: k for k, v in char_to_idx.items() if k != '<BLANK>'}
    seq_len, batch_size, num_classes = predictions.shape

    decoded_texts = []

    for b in range(batch_size):
        # Initialize beam: (prefix, prob, last_char)
        beam = [("", 0.0, -1)]

        for t in range(seq_len):
            log_probs = torch.log_softmax(predictions[t, b], dim=0).cpu().numpy()
            new_beam = []

            for prefix, prob, last_char in beam:
                # Option 1: emit blank
                new_beam.append((prefix, prob + log_probs[blank_index], -1))

                # Option 2: emit non-blank characters
                for c in range(num_classes):
                    if c == blank_index:
                        continue
                    char = idx_to_char.get(c, '')
                    if c == last_char:
                        # Merge repeated characters
                        new_beam.append((prefix, prob + log_probs[c], c))
                    else:
                        new_beam.append((prefix + char, prob + log_probs[c], c))

            # Keep top-k beams
            new_beam.sort(key=lambda x: x[1], reverse=True)
            beam = new_beam[:beam_width]

        decoded_texts.append(beam[0][0])

    return decoded_texts

In [ ]:
def calculate_accuracy(decoded_preds, true_labels):
    """Character-level and sequence-level accuracy"""
    correct_chars = 0
    total_chars = 0
    correct_sequences = 0

    for pred, true in zip(decoded_preds, true_labels):
        total_chars += len(true)
        correct_chars += sum(p == t for p, t in zip(pred, true))
        if pred == true:
            correct_sequences += 1

    char_acc = correct_chars / total_chars if total_chars > 0 else 0
    seq_acc = correct_sequences / len(true_labels) if len(true_labels) > 0 else 0

    return char_acc, seq_acc

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                char_to_idx, num_epochs=50, device='cpu'):

    best_seq_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for batch_idx, (images, targets, target_lengths) in enumerate(train_loader):
            images = images.to(device)
            targets = targets.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            outputs = model(images)  # (seq_len, batch, num_classes)

            input_lengths = torch.full(
                size=(outputs.size(1),),
                fill_value=outputs.size(0),
                dtype=torch.long
            ).to(device)

            loss = criterion(outputs, targets, input_lengths, target_lengths)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_preds = []
        val_labels = []
        val_loss = 0.0

        with torch.no_grad():
            for images, targets, target_lengths in val_loader:
                images = images.to(device)
                targets = targets.to(device)
                target_lengths = target_lengths.to(device)

                outputs = model(images)
                input_lengths = torch.full(
                    size=(outputs.size(1),),
                    fill_value=outputs.size(0),
                    dtype=torch.long
                ).to(device)

                loss = criterion(outputs, targets, input_lengths, target_lengths)
                val_loss += loss.item()

                # Decode predictions
                decoded = ctc_greedy_decode(outputs.cpu(), char_to_idx)
                val_preds.extend(decoded)
                val_labels.extend([label for label in targets_batch_to_strings(targets, target_lengths, char_to_idx)])

        char_acc, seq_acc = calculate_accuracy(val_preds, val_labels)

        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'  Train Loss: {train_loss/len(train_loader):.4f}')
        print(f'  Val Loss: {val_loss/len(val_loader):.4f}')
        print(f'  Char Acc: {char_acc:.4f}, Seq Acc: {seq_acc:.4f}')

        # Save best model
        if seq_acc > best_seq_acc:
            best_seq_acc = seq_acc
            torch.save(model.state_dict(), 'best_crnn_captcha-2.pth')
            print(f'  → New best model saved! (Seq Acc: {seq_acc:.4f})')

In [ ]:
def targets_batch_to_strings(targets, lengths, char_to_idx):
    idx_to_char = {v: k for k, v in char_to_idx.items() if k != '<BLANK>'}
    strings = []
    start = 0
    for length in lengths:
        end = start + length
        label_indices = targets[start:end].cpu().numpy()
        string = ''.join([idx_to_char[idx] for idx in label_indices])
        strings.append(string)
        start = end
    return strings

In [ ]:
def predict_captcha(model, image_path, char_to_idx, device='cpu'):
    """
    Predict CAPTCHA from a single 200x50 image
    """
    model.eval()

    # Preprocess image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    img = cv2.resize(img, (200, 50), interpolation=cv2.INTER_CUBIC)
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5  # Normalize to [-1, 1]
    img_tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,50,200)

    with torch.no_grad():
        output = model(img_tensor)  # (51, 1, num_classes)
        prediction = ctc_greedy_decode(output, char_to_idx)[0]

    return prediction

# Example usage
# model.load_state_dict(torch.load('best_crnn_captcha.pth', map_location=device))
# predicted_text = predict_captcha(model, 'test_captcha.png', char_to_idx, device)
# print(f"Predicted CAPTCHA: {predicted_text}")

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.3, contrast=0.3),
    ], p=0.5),
    transforms.RandomApply([
        transforms.Lambda(lambda x: add_gaussian_noise(x)),
    ], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

def add_gaussian_noise(img, mean=0, var=0.01):
    img = np.array(img) / 255.0
    noise = np.random.normal(mean, var ** 0.5, img.shape)
    img = np.clip(img + noise, 0, 1)
    return (img * 255).astype(np.uint8)

In [ ]:
# full_train.py
from torch.utils.data import DataLoader
import torch
import os

# Config
IMG_H, IMG_W = 50, 200
CHARSET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%"
char_to_idx = {'<BLANK>': 0}
char_to_idx.update({c: i+1 for i, c in enumerate(CHARSET)})
NUM_CLASSES = len(char_to_idx)






# Create synthetic data
# generator = CAPTCHAGenerator(width=IMG_W, height=IMG_H, char_set=CHARSET)
# train_dataset = SyntheticCAPTCHADataset(generator, size=50000, char_to_idx=char_to_idx)
# val_dataset = SyntheticCAPTCHADataset(generator, size=5000, char_to_idx=char_to_idx)


# Example usage
# image_paths = ["captcha_1.png", "captcha_2.png", ...]
# labels = ["A3B9", "X7Y2Z", ...]

def custom_collate_fn(batch):
    images, labels, lengths = zip(*batch)
    images = torch.stack(images, 0)
    labels = torch.cat(labels, 0)
    lengths = torch.LongTensor(lengths)
    return images, labels, lengths


image_paths = [os.path.join("captcha",x) for x in os.listdir("captcha")]
labels = [x.split(".")[0] for x in os.listdir("captcha")]

train_dataset = CAPTCHADataset(image_paths, labels, char_to_idx)


image_paths_val = [os.path.join("captcha_test",x) for x in os.listdir("captcha_test")]
labels_val = [x.split(".")[0] for x in os.listdir("captcha_test")]


val_dataset = CAPTCHADataset(image_paths_val, labels_val, char_to_idx)

# dataloader = DataLoader(
#     dataset,
#     batch_size=32,
#     shuffle=True,
#     collate_fn=custom_collate_fn  # Handle variable length labels
# )


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=custom_collate_fn)

# Model & training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CRNN(img_h=IMG_H, img_w=IMG_W, num_classes=NUM_CLASSES).to(device)
criterion = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)



In [ ]:
image_paths = [os.path.join("captcha",x) for x in os.listdir("captcha")]
print(image_paths)

['captcha/n245a.png', 'captcha/n2apkk.png', 'captcha/n2db55.png', 'captcha/n2e7aa.png', 'captcha/n2kcc.png', 'captcha/n2p4ww.png', 'captcha/n2xakk.png', 'captcha/n2y8h.png', 'captcha/n383m.png', 'captcha/n38db.png', 'captcha/n3c7hh.png', 'captcha/n3dya.png', 'captcha/n3pbkk.png', 'captcha/n3yab.png', 'captcha/n3yk6.png', 'captcha/n43rx.png', 'captcha/n45gd.png', 'captcha/n4byg.png', 'captcha/n4caxx.png', 'captcha/n4ebn.png', 'captcha/n4hdc.png', 'captcha/n4kre.png', 'captcha/n4n6yy.png', 'captcha/n4r7aa.png', 'captcha/n54ae.png', 'captcha/n55k66.png', 'captcha/n57ax.png', 'captcha/n5a56.png', 'captcha/n5br88.png', 'captcha/n5dng.png', 'captcha/n5dxx.png', 'captcha/n5g27.png', 'captcha/n5gw5.png', 'captcha/n5hrgg.png', 'captcha/n5kfff.png', 'captcha/n5pf44.png', 'captcha/n5ydm.png', 'captcha/n62ngg.png', 'captcha/n642g.png', 'captcha/n6d4f.png', 'captcha/n6fd55.png', 'captcha/n6hwb.png', 'captcha/n6pr5.png', 'captcha/n6r8ee.png', 'captcha/n72xw.png', 'captcha/n74ag.png', 'captcha/n74fn.

In [ ]:
# Train
train_model(model, train_loader, val_loader, criterion, optimizer, char_to_idx,
            num_epochs=30, device=device)

KeyError: '_'

In [ ]:
# 🖼️ 4. Inference Pipeline
def predict_captcha(model, image_path, char_to_idx, device='cpu'):
    """
    Predict CAPTCHA from a single 200x50 image
    """
    model.eval()

    # Preprocess image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    img = cv2.resize(img, (200, 50), interpolation=cv2.INTER_CUBIC)
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5  # Normalize to [-1, 1]
    img_tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,50,200)

    with torch.no_grad():
        output = model(img_tensor)  # (51, 1, num_classes)
        prediction = ctc_greedy_decode(output, char_to_idx)[0]

    return prediction

# Example usage
print(image_paths_val[3])
model.load_state_dict(torch.load('best_crnn_captcha.pth', map_location=device))
predicted_text = predict_captcha(model, image_paths_val[3], char_to_idx, device)
print(f"Predicted CAPTCHA: {predicted_text}")

captcha_test/2efxw.png
Predicted CAPTCHA: 2efxw


In [ ]:
!ls

best_crnn_captcha.pth  captcha_crnn.onnx  captch_sample.zip  map_data.json
captcha		       captcha_test	  correctImage	     out.png


In [ ]:
def export_to_onnx(model, save_path="captcha_crnn.onnx"):
    model.eval()

    # Move to CPU to avoid device mismatch
    model_cpu = model.cpu()
    dummy_input = torch.randn(1, 1, 50, 200)  # (B, C, H, W)

    # Use LEGACY exporter (no dynamo)
    torch.onnx.export(
        model_cpu,
        dummy_input,
        save_path,
        export_params=True,
        opset_version=13,  # Stable for RNNs + CNNs
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "output": {1: "batch_size"}  # CTC output: (seq_len, batch, classes)
        },
        # ⚠️ DO NOT set dynamo=True
    )
    print(f"✅ Successfully exported to {save_path}")

export_to_onnx(model)

In [ ]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.2 MB/s eta 0:00:00


In [ ]:
import onnxruntime as ort
import numpy as np
import cv2

def preprocess_image(image_path, width=200, height=50):
    """Preprocess image to match model input (1, 1, 50, 200)"""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")

    img = cv2.resize(img, (width, height), interpolation=cv2.INTER_CUBIC)
    img = img.astype(np.float32) / 255.0          # [0, 1]
    img = (img - 0.5) / 0.5                       # [-1, 1] (same as training)
    img = img[np.newaxis, np.newaxis, :, :]       # (1, 1, 50, 200)
    return img

# Load ONNX model
ort_session = ort.InferenceSession("captcha_crnn.onnx")

# Get input/output names
input_name = ort_session.get_inputs()[0].name
output_name = ort_session.get_outputs()[0].name

print(f"Input name: {input_name}, shape: {ort_session.get_inputs()[0].shape}")
print(f"Output name: {output_name}, shape: {ort_session.get_outputs()[0].shape}")

Input name: input, shape: ['batch_size', 1, 50, 200]
Output name: output, shape: [51, 'batch_size', 68]


In [ ]:
def ctc_decode_onnx(predictions, char_to_idx, blank_index=0):
    """Greedy decode CTC output from ONNX model"""
    idx_to_char = {v: k for k, v in char_to_idx.items() if k != '<BLANK>'}
    _, max_indices = np.argmax(predictions, axis=2), np.max(predictions, axis=2)

    # predictions shape: (seq_len, batch, num_classes)
    decoded = []
    for b in range(predictions.shape[1]):
        prev_idx = -1
        text = ""
        for t in range(predictions.shape[0]):
            idx = int(np.argmax(predictions[t, b]))
            if idx != blank_index and idx != prev_idx:
                text += idx_to_char.get(idx, '')
            prev_idx = idx
        decoded.append(text)
    return decoded

# Example usage
print(image_paths_val[11])
image_tensor = preprocess_image(image_paths_val[11])  # (1, 1, 50, 200)
outputs = ort_session.run([output_name], {input_name: image_tensor})
logits = outputs[0]  # Shape: (seq_len, 1, num_classes)

# Decode
CHARSET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%"
char_to_idx = {'<BLANK>': 0}
char_to_idx.update({c: i+1 for i, c in enumerate(CHARSET)})

predicted_text = ctc_decode_onnx(logits, char_to_idx)[0]
print(f"Predicted CAPTCHA: {predicted_text}")

captcha_test/y66y3.png
Predicted CAPTCHA: y66ya
